In [1]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check GPU status
!nvidia-smi

Mon Jul 13 16:28:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libraries

In [3]:
%%capture
!uv pip install evaluate

In [4]:
import json
import torch
import evaluate
import pandas as pd
from datetime import datetime
from pathlib import Path
from transformers import (
    AutoTokenizer, 
    AutoModelForQuestionAnswering, 
    DataCollatorWithPadding, 
    Trainer, 
    TrainingArguments,
)
from datasets import load_dataset, Dataset

# Configurations

In [ ]:
# Run configuration
SEED = 42
LANGUAGES = ['en', 'ar', 'de', 'el', 'es', 'hi', 'ru', 'th', 'tr', 'vi', 'zh']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Model configuration
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-en-1K-LoRA-Merged-v260623145250'
MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-vi-1K-LoRA-Addition-v260713162325'
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-Merged-v260711104723'
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-vi-5K-LoRA-Addition-v260712125532'

# Data configuration
TEST_SIZE = 125
# TEST_SIZE = 625
DATA_ID = 'google/xquad'
DATA_DIR = 'xquad.{lang}'
DATA_SPLIT = 'validation'

# Evaluation configuration
BATCH_SIZE = 16

# Set up the evaluation directory
eval_dir = f'./eval/xquad_xlmr/{TEST_SIZE}/{MODEL_ID}'
print(f"Evaluation directory: {eval_dir}")

Evaluation directory: ./eval/xquad_xlmr/125/alxxtexxr/XLM-R-Base-squad-vi-1K-LoRA-Addition-v260713162325


# Utilities

In [6]:
def load_test_dataset(
    lang, # e.g., 'en' | 'ja' | 'id'
    size,
    data_id=DATA_ID,
    data_dir=DATA_DIR,
    data_split=DATA_SPLIT,
):
    assert '{lang}' in data_dir, "Data directory must contain a '{lang}' placeholder."
    
    dataset_stream = load_dataset(
        data_id,
        data_dir=data_dir.format(lang=lang),
        split=data_split,
        streaming=True,
    )

    test_data = []

    for i, example in enumerate(dataset_stream):
        if i < size:
            test_data.append(example)
        else:
            break

    return Dataset.from_list(test_data)

# Model

In [7]:
# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_ID)
model.eval().to(DEVICE)

print("device:", model.device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


device: cuda:0


# Data

In [8]:
# Preprocess the test dataset for evaluation
def preprocess_squad(examples):
    tokenized = tokenizer(
        examples['question'],
        examples['context'],
        truncation='only_second',
        max_length=384,
        return_offsets_mapping=True,
    )
    start_positions = []
    end_positions = []
    for i, offsets in enumerate(tokenized['offset_mapping']):
        sequence_ids = tokenized.sequence_ids(i)
        answer = examples['answers'][i]
        answer_start_char = answer['answer_start'][0]
        answer_text = answer['text'][0]
        answer_end_char = answer_start_char + len(answer_text)

        token_start = None
        token_end = None
        for idx, (offset_start, offset_end) in enumerate(offsets):
            if sequence_ids[idx] != 1:
                continue
            if offset_start >= answer_start_char and offset_end <= answer_end_char:
                if token_start is None:
                    token_start = idx
                token_end = idx
            elif offset_start < answer_end_char and offset_end > answer_start_char:
                if token_start is None:
                    token_start = idx
                token_end = idx

        if token_start is None or token_end is None:
            token_start = 0
            token_end = 0

        start_positions.append(token_start)
        end_positions.append(token_end)

    tokenized['start_positions'] = start_positions
    tokenized['end_positions'] = end_positions
    return tokenized

# Evaluation

In [9]:
# Set up the trainer for evaluation
data_collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)
eval_args = TrainingArguments(
    output_dir=eval_dir,
    do_train=False,
    do_eval=True,
    per_device_eval_batch_size=BATCH_SIZE,
    dataloader_drop_last=False,
    report_to=[],
)
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=eval_args,
)
trainer.label_names = ['start_positions', 'end_positions']

In [10]:
results = {}
squad_metric = evaluate.load('squad')

for i, lang in enumerate(LANGUAGES):
    print(f"\n{'=' * 64}")
    print(f"[{i+1}/{len(LANGUAGES)}] Evaluating: {lang}")
    print(f"{'=' * 64}")
    
    # Load and preprocess the dataset for this language
    raw_dataset = load_test_dataset(lang, size=TEST_SIZE)
    test_dataset = raw_dataset.map(
        preprocess_squad, 
        batched=True, 
        remove_columns=raw_dataset.column_names
    )
    
    # Create a closure that captures this language's datasets
    def make_compute_metrics(raw_ds, test_ds):
        def compute_metrics_lang(pred):
            start_logits, end_logits = pred.predictions
            
            predictions = []
            for i, (start_log, end_log) in enumerate(zip(start_logits, end_logits)):
                start_idx = start_log.argmax()
                end_idx = end_log.argmax()
                input_ids = test_ds[i]['input_ids']
                answer_ids = input_ids[start_idx : end_idx + 1]
                answer_text = tokenizer.decode(answer_ids, skip_special_tokens=True)
                predictions.append({
                    'id': raw_ds[i]['id'],
                    'prediction_text': answer_text,
                })
            
            references = []
            for example in raw_ds:
                references.append({
                    'id': example['id'],
                    'answers': {
                        'text': example['answers']['text'],
                        'answer_start': example['answers']['answer_start'],
                    },
                })
            
            return squad_metric.compute(predictions=predictions, references=references)
        return compute_metrics_lang
    
    # Update trainer's compute_metrics for this language
    trainer.compute_metrics = make_compute_metrics(raw_dataset, test_dataset)
    
    # Run prediction
    predictions = trainer.predict(test_dataset)
    
    # Store results
    results[lang] = predictions.metrics
    
    print(f"Exact Match: {results[lang]['test_exact_match']:.2f}%")
    print(f"F1: {results[lang]['test_f1']:.2f}%")
    
    # Save per-language predictions
    pred_path = f"{eval_dir}/predictions_{lang}.txt"
    start_logits, end_logits = predictions.predictions
    with open(pred_path, 'w', encoding='utf-8') as f:
        for i in range(len(test_dataset)):
            start_idx = start_logits[i].argmax()
            end_idx = end_logits[i].argmax()
            input_ids = test_dataset[i]['input_ids']
            answer_ids = input_ids[start_idx:end_idx + 1]
            pred_text = tokenizer.decode(answer_ids, skip_special_tokens=True)
            gt_text = raw_dataset[i]['answers']['text'][0]
            
            f.write(f"Q: {raw_dataset[i]['question']}\n")
            f.write(f"GT: {gt_text}\n")
            f.write(f"Pred: {pred_text}\n")
            f.write(f"Match: {pred_text.strip().lower() == gt_text.strip().lower()}\n")
            f.write("-" * 64 + "\n")
    
    print("-" * 64)
    print(f"Saved predictions to: {pred_path}")

# Print summary table
print(f"\n{'=' * 64}")
print("Evaluation Result Overview")
print(f"{'=' * 64}")
print(f"{'Language':<10} {'Exact Match':>12} {'F1':>12}")
print("-" * 36)
for lang in LANGUAGES:
    print(f"{lang:<10} {results[lang]['test_exact_match']:>11.2f}% {results[lang]['test_f1']:>11.2f}%")

# Calculate averages
avg_em = sum(r['test_exact_match'] for r in results.values()) / len(results)
avg_f1 = sum(r['test_f1'] for r in results.values()) / len(results)
print("-" * 36)
print(f"{'Average':<10} {avg_em:>11.2f}% {avg_f1:>11.2f}%")


[1/11] Evaluating: en


Map:   0%|          | 0/125 [00:00<?, ? examples/s]

Exact Match: 52.80%
F1: 59.44%
----------------------------------------------------------------
Saved predictions to: ./eval/xquad_xlmr/125/alxxtexxr/XLM-R-Base-squad-vi-1K-LoRA-Addition-v260713162325/predictions_en.txt

[2/11] Evaluating: ar


Map:   0%|          | 0/125 [00:00<?, ? examples/s]

Exact Match: 36.80%
F1: 42.50%
----------------------------------------------------------------
Saved predictions to: ./eval/xquad_xlmr/125/alxxtexxr/XLM-R-Base-squad-vi-1K-LoRA-Addition-v260713162325/predictions_ar.txt

[3/11] Evaluating: de


...

Exact Match: 38.40%
F1: 46.00%
----------------------------------------------------------------
Saved predictions to: ./eval/xquad_xlmr/125/alxxtexxr/XLM-R-Base-squad-vi-1K-LoRA-Addition-v260713162325/predictions_tr.txt

[10/11] Evaluating: vi


Map:   0%|          | 0/125 [00:00<?, ? examples/s]

Exact Match: 37.60%
F1: 48.87%
----------------------------------------------------------------
Saved predictions to: ./eval/xquad_xlmr/125/alxxtexxr/XLM-R-Base-squad-vi-1K-LoRA-Addition-v260713162325/predictions_vi.txt

[11/11] Evaluating: zh


Map:   0%|          | 0/125 [00:00<?, ? examples/s]

Exact Match: 40.00%
F1: 45.03%
----------------------------------------------------------------
Saved predictions to: ./eval/xquad_xlmr/125/alxxtexxr/XLM-R-Base-squad-vi-1K-LoRA-Addition-v260713162325/predictions_zh.txt

Evaluation Result Overview
Language    Exact Match           F1
------------------------------------
en               52.80%       59.44%
ar               36.80%       42.50%
de               36.00%       48.51%
el               40.00%       49.38%
es               43.20%       49.63%
hi               36.00%       45.66%
ru               36.00%       46.76%
th               40.00%       51.59%
tr               38.40%       46.00%
vi               37.60%       48.87%
zh               40.00%       45.03%
------------------------------------
Average          39.71%       48.49%


In [11]:
# Save metrics JSON and CSV
metrics_json_path = f'{eval_dir}/metrics.json'
metrics_csv_path = f'{eval_dir}/metrics.csv'

metrics = []
for lang in LANGUAGES:
    metrics.append({
        'lang': lang,
        # 'exact_match': results[lang]['exact_match'],
        # 'f1': results[lang]['f1'],
        **results[lang],
    })
metrics_df = pd.DataFrame(metrics)
metrics_df = metrics_df[[
    'lang', 'test_loss', 'test_exact_match', 'test_f1', 
    'test_model_preparation_time', 'test_runtime', 
    'test_samples_per_second', 'test_steps_per_second'
]] # Rearrange columns

metrics_df.to_json(metrics_json_path, orient='records')
metrics_df.to_csv(metrics_csv_path, index=False)

print(f"Saved metrics JSON to: {metrics_json_path}")
print(f"Saved metrics CSV to: {metrics_csv_path}")

Saved metrics JSON to: ./eval/xquad_xlmr/125/alxxtexxr/XLM-R-Base-squad-vi-1K-LoRA-Addition-v260713162325/metrics.json
Saved metrics CSV to: ./eval/xquad_xlmr/125/alxxtexxr/XLM-R-Base-squad-vi-1K-LoRA-Addition-v260713162325/metrics.csv


In [12]:
# Save metadata JSON
metadata_json_path = f'{eval_dir}/metadata.json'
metadata = {
    'model_id': MODEL_ID,
    'data_id': DATA_ID,
    'data_dir': DATA_DIR,
    'data_split': DATA_SPLIT,
    'test_size': TEST_SIZE,
    'batch_size': BATCH_SIZE,
    'evaluated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
}

with open(metadata_json_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=4)
    
print(f"Saved metadata JSON to: {metadata_json_path}")

Saved metadata JSON to: ./eval/xquad_xlmr/125/alxxtexxr/XLM-R-Base-squad-vi-1K-LoRA-Addition-v260713162325/metadata.json


In [13]:
# Optional: Create a zip file of the 'eval' directory
zip_dir = str(Path(eval_dir).parent.parent)
zip_name = zip_dir.replace('/', '_').replace('\\', '_')
import shutil; shutil.make_archive(zip_name, 'zip', zip_dir)
print(f"Created zip file: {zip_name}.zip")

Created zip file: eval_xquad_xlmr_125.zip
